# Лабораторная работа № 7

## Модель конкуренции двух фирм

Вариант 58.

Исследуются два случая конкурентной борьбы:
только экономические факторы и модель с дополнительным
социально-психологическим воздействием.

In [1]:
using DrWatson
@quickactivate "project"

using DifferentialEquations
using Plots
using DataFrames
using CSV

## Initial data

In [2]:
M10 = 7.2
M20 = 9.1

p_cr = 42.0
N = 45.0
q = 1.0

tau1 = 28.0
tau2 = 22.0

p1 = 8.1
p2 = 10.5

u0 = [M10, M20]

mkpath("data")
mkpath("plots")

"plots"

## Model coefficients

In [3]:
a1 =
    p_cr /
    (tau1^2 * p1^2 * N * q)

a2 =
    p_cr /
    (tau2^2 * p2^2 * N * q)

b =
    p_cr /
    (
        tau1^2 *
        tau2^2 *
        p1^2 *
        p2^2 *
        N *
        q
    )

c1 =
    (p_cr - p1) /
    (tau1 * p1)

c2 =
    (p_cr - p2) /
    (tau2 * p2)

println("Model coefficients:")
println("a1 = ", a1)
println("a2 = ", a2)
println("b  = ", b)
println("c1 = ", c1)
println("c2 = ", c2)

Model coefficients:
a1 = 1.814473693760388e-5
a2 = 1.7490926581835673e-5
b  = 3.4003742316680497e-10
c1 = 0.14947089947089948
c2 = 0.13636363636363635


## Case 1

Only economic competition factors are considered.

dM1/dtheta =
M1 - (b/c1) M1 M2 - (a1/c1) M1^2

dM2/dtheta =
(c2/c1) M2 - (b/c1) M1 M2 - (a2/c1) M2^2

In [4]:
function firms_case1!(du, u, p, theta)
    M1, M2 = u

    du[1] =
        M1 -
        (b / c1) * M1 * M2 -
        (a1 / c1) * M1^2

    du[2] =
        (c2 / c1) * M2 -
        (b / c1) * M1 * M2 -
        (a2 / c1) * M2^2
end

problem1 = ODEProblem(
    firms_case1!,
    u0,
    (0.0, 30.0),
)

solution1 = solve(
    problem1,
    Tsit5();
    saveat=0.05,
    abstol=1e-9,
    reltol=1e-9,
)

theta1 = solution1.t
M1_case1 = getindex.(solution1.u, 1)
M2_case1 = getindex.(solution1.u, 2)

601-element Vector{Float64}:
    9.1
    9.524194819047963
    9.968138088880108
   10.43274678702069
   10.91898023836812
   11.427842049493055
   11.960382128004376
   12.517698792466058
   13.100940975847745
   13.711310526319169
   14.350064609734622
   15.018518217807815
   15.718046786498743
    ⋮
 7796.089841077423
 7796.089841717498
 7796.089842328473
 7796.089842911719
 7796.089843468563
 7796.089844000304
 7796.089844508198
 7796.08984499347
 7796.089845457308
 7796.08984590086
 7796.089846325244
 7796.089846731538

## Stationary state for case 1

For positive M1 and M2:

a1*M1 + b*M2 = c1

b*M1 + a2*M2 = c2

In [5]:
determinant =
    a1 * a2 - b^2

M1_stationary =
    (c1 * a2 - b * c2) /
    determinant

M2_stationary =
    (a1 * c2 - b * c1) /
    determinant

println()
println("Case 1 stationary state:")
println("M1* = ", M1_stationary)
println("M2* = ", M2_stationary)


Case 1 stationary state:
M1* = 8237.553899105049
M2* = 7796.089855448025


## Case 2

Additional social-psychological influence is introduced.

In [6]:
social_factor = 0.00048

function firms_case2!(du, u, p, theta)
    M1, M2 = u

    du[1] =
        M1 -
        (b / c1 + social_factor) *
        M1 * M2 -
        (a1 / c1) * M1^2

    du[2] =
        (c2 / c1) * M2 -
        (b / c1) * M1 * M2 -
        (a2 / c1) * M2^2
end

problem2 = ODEProblem(
    firms_case2!,
    u0,
    (0.0, 30.0),
)

solution2 = solve(
    problem2,
    Tsit5();
    saveat=0.05,
    abstol=1e-9,
    reltol=1e-9,
)

theta2 = solution2.t
M1_case2 = getindex.(solution2.u, 1)
M2_case2 = getindex.(solution2.u, 2)

601-element Vector{Float64}:
    9.1
    9.52419481904883
    9.96813808888395
   10.43274678703383
   10.918980238390725
   11.42784204952046
   11.960382128048298
   12.517698792541
   13.100940975954021
   13.711310526467681
   14.350064609919576
   15.018518218046772
   15.718046786833488
    ⋮
 7796.249985637582
 7796.2499862768345
 7796.249986886919
 7796.249987469226
 7796.249988025114
 7796.249988555908
 7796.249989062899
 7796.249989547343
 7796.249990010465
 7796.249990453455
 7796.24999087747
 7796.249991283631

## Save data

In [7]:
CSV.write(
    "data/firms_case1.csv",
    DataFrame(
        theta=theta1,
        firm1=M1_case1,
        firm2=M2_case1,
    ),
)

CSV.write(
    "data/firms_case2.csv",
    DataFrame(
        theta=theta2,
        firm1=M1_case2,
        firm2=M2_case2,
    ),
)

"data/firms_case2.csv"

## Case 1 plot

In [8]:
p_case1 = plot(
    theta1,
    M1_case1;
    label="Firm 1",
    linewidth=2,
    xlabel="Normalized time",
    ylabel="Working capital",
    title="Competition of firms: case 1",
)

plot!(
    p_case1,
    theta1,
    M2_case1;
    label="Firm 2",
    linewidth=2,
)

savefig(
    p_case1,
    "plots/firms_case1.png",
)

"/home/adsvettsova/work/study/2026-1/2026-1--study-mathematical-modeling/labs/lab07/project/notebooks/01_firms/plots/firms_case1.png"

## Case 2 plot

In [9]:
p_case2 = plot(
    theta2,
    M1_case2;
    label="Firm 1",
    linewidth=2,
    xlabel="Normalized time",
    ylabel="Working capital",
    title="Competition of firms: case 2",
)

plot!(
    p_case2,
    theta2,
    M2_case2;
    label="Firm 2",
    linewidth=2,
)

savefig(
    p_case2,
    "plots/firms_case2.png",
)

"/home/adsvettsova/work/study/2026-1/2026-1--study-mathematical-modeling/labs/lab07/project/notebooks/01_firms/plots/firms_case2.png"

## Phase portrait for case 1

In [10]:
p_phase = plot(
    M1_case1,
    M2_case1;
    label="Trajectory",
    linewidth=2,
    xlabel="Firm 1",
    ylabel="Firm 2",
    title="Case 1: phase trajectory",
)

scatter!(
    p_phase,
    [M1_stationary],
    [M2_stationary];
    label="Stationary state",
    markersize=6,
)

savefig(
    p_phase,
    "plots/firms_case1_phase.png",
)

"/home/adsvettsova/work/study/2026-1/2026-1--study-mathematical-modeling/labs/lab07/project/notebooks/01_firms/plots/firms_case1_phase.png"

## Comparison

In [11]:
p_compare = plot(
    theta1,
    M1_case1;
    label="Firm 1, case 1",
    linewidth=2,
    xlabel="Normalized time",
    ylabel="Working capital",
    title="Comparison of competition models",
)

plot!(
    p_compare,
    theta1,
    M2_case1;
    label="Firm 2, case 1",
    linewidth=2,
)

plot!(
    p_compare,
    theta2,
    M1_case2;
    label="Firm 1, case 2",
    linewidth=2,
    linestyle=:dash,
)

plot!(
    p_compare,
    theta2,
    M2_case2;
    label="Firm 2, case 2",
    linewidth=2,
    linestyle=:dash,
)

savefig(
    p_compare,
    "plots/firms_comparison.png",
)

println()
println("Final values:")
println(
    "Case 1: M1 = ",
    last(M1_case1),
    ", M2 = ",
    last(M2_case1),
)

println(
    "Case 2: M1 = ",
    last(M1_case2),
    ", M2 = ",
    last(M2_case2),
)

println()
println("LAB07 BASE OK")


Final values:
Case 1: M1 = 8237.553898215332, M2 = 7796.089846731538
Case 2: M1 = 2.0369634488397234e-18, M2 = 7796.249991283631

LAB07 BASE OK
